# 09 · Inspect the project database

Run the first Airflow DAG before querying the tables.

## Imports

In [1]:
import os
import subprocess
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import text

load_dotenv(Path('../.env'))
if not os.getenv('APP_DB_HOST'):
    kubeconfig = os.getenv('KUBECONFIG') or str(Path.home() / '.kube/project-3-k3s.yaml')
    os.environ['APP_DB_HOST'] = subprocess.check_output([
        'kubectl', '--kubeconfig', kubeconfig, '--request-timeout=10s', '-n', 'project-3',
        'get', 'svc', 'property-postgres', '-o', 'jsonpath={.spec.clusterIP}',
    ], text=True).strip()

sys.path.insert(0, str(Path('../src').resolve()))
from property_ml.settings import engine

db = engine()

## Batches

In [2]:
with db.connect() as conn:
    display(pd.read_sql_query(text('SELECT * FROM raw.batches ORDER BY group_number, batch_number'), conn))


,group_number,batch_number,run_id,received_at,row_count,status,received_columns,valid_rows,rejected_rows,duplicate_rows,quality


## Raw and clean counts

In [3]:
with db.connect() as conn:
    for table in ['raw.properties', 'clean.properties', 'audit.inference_events']:
        print(table, conn.execute(text(f'SELECT count(*) FROM {table}')).scalar())


raw.properties 0
clean.properties 0
audit.inference_events 0


## Decisions and model history

In [4]:
with db.connect() as conn:
    display(pd.read_sql_query(text('SELECT * FROM audit.batch_decisions ORDER BY batch_number'), conn))
    display(pd.read_sql_query(text('SELECT * FROM audit.training_events ORDER BY id'), conn))


,group_number,batch_number,created_at,valid_rows,pending_rows,decision,reason,model_version,candidate_mae,champion_mae,promotion,run_id,metrics,report_key


,id,group_number,batch_number,created_at,mlflow_run_id,model_version,candidate_mae,champion_mae,promoted,reason,train_rows


## Sample predictions

In [5]:
with db.connect() as conn:
    display(pd.read_sql_query(text('SELECT created_at, prediction, model_version, pod, status, error FROM audit.inference_events ORDER BY id DESC LIMIT 10'), conn))


,created_at,prediction,model_version,pod,status,error


## Training data coverage

In [6]:
with db.connect() as conn:
    display(pd.read_sql_query(text('''
        WITH available AS (
            SELECT group_number, count(*) AS clean_rows
            FROM clean.properties GROUP BY group_number
        ), champions AS (
            SELECT DISTINCT ON (group_number) group_number, batch_number, model_version, (to_jsonb(t)->>'train_rows')::int AS train_rows
            FROM audit.training_events t WHERE promoted
            ORDER BY group_number, batch_number DESC
        )
        SELECT a.group_number, a.clean_rows, c.batch_number, c.model_version, c.train_rows,
               round(100.0 * c.train_rows / nullif(a.clean_rows, 0), 1) AS train_pct_of_current_clean
        FROM available a LEFT JOIN champions c USING (group_number)
    '''), conn))
db.dispose()


,group_number,clean_rows,batch_number,model_version,train_rows,train_pct_of_current_clean
